# 工程工具与应用

各节为独立示例，可按需阅读和运行。

目录：

- APScheduler
- Pydantic 数据校验工具
- Gradio 和 Streamlit 选择
- 可测试的业务函数
- Gradio
- Streamlit
- Python Web 应用项目结构
- Playwright 浏览器自动化

## APScheduler

APScheduler 是应用级定时任务工具，适合后台同步、报表生成、定时清理和轻量任务编排。它支持 interval、date、cron 等触发器，以及线程池/进程池执行器。

### 基本配置

In [ ]:
from apscheduler.executors.pool import ProcessPoolExecutor, ThreadPoolExecutor
from apscheduler.jobstores.memory import MemoryJobStore
from apscheduler.schedulers.background import BackgroundScheduler

scheduler = BackgroundScheduler(
    timezone="Asia/Shanghai",
    jobstores={"default": MemoryJobStore()},
    executors={
        "default": ThreadPoolExecutor(max_workers=10),
        "processpool": ProcessPoolExecutor(max_workers=2),
    },
    job_defaults={
        "coalesce": True,
        "max_instances": 1,
        "misfire_grace_time": 30,
    },
)

### Trigger 示例

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from apscheduler.triggers.cron import CronTrigger
from apscheduler.triggers.date import DateTrigger
from apscheduler.triggers.interval import IntervalTrigger

SHANGHAI = ZoneInfo("Asia/Shanghai")


def send_report(report_name: str) -> str:
    return f"{report_name} sent at {datetime.now(SHANGHAI):%Y-%m-%d %H:%M:%S}"


scheduler.add_job(
    send_report,
    trigger=IntervalTrigger(seconds=10),
    args=["interval-report"],
    id="interval_report",
    replace_existing=True,
)

scheduler.add_job(
    send_report,
    trigger=DateTrigger(run_date=datetime.now(SHANGHAI) + timedelta(minutes=5)),
    args=["once-report"],
    id="once_report",
    replace_existing=True,
)

scheduler.add_job(
    send_report,
    trigger=CronTrigger(hour=9, minute=30, timezone=SHANGHAI),
    args=["daily-report"],
    id="daily_report",
    replace_existing=True,
)

### 管理任务

In [ ]:
def refresh_cache(cache_name: str) -> str:
    return f"{cache_name} refreshed"


scheduler.add_job(
    refresh_cache, "interval", seconds=30, args=["user_profile"], id="refresh_cache"
)
job_ref = scheduler.get_job("refresh_cache")
print("Job:", job_ref)

scheduler.modify_job("refresh_cache", seconds=60)
scheduler.pause_job("refresh_cache")
scheduler.resume_job("refresh_cache")
scheduler.remove_job("refresh_cache")

### 监听器与生命周期

In [ ]:
from apscheduler.events import EVENT_JOB_ERROR, EVENT_JOB_EXECUTED, JobExecutionEvent


def log_job_event(event: JobExecutionEvent) -> None:
    if event.exception:
        print(f"Job failed: {event.job_id}")
    else:
        print(f"Job succeeded: {event.job_id}")


scheduler.add_listener(log_job_event, EVENT_JOB_EXECUTED | EVENT_JOB_ERROR)
scheduler.print_jobs()

if not scheduler.running:
    scheduler.start(paused=True)

scheduler.resume()
scheduler.pause()
scheduler.shutdown(wait=False)

### 应用级封装

In [ ]:
from collections.abc import Callable

from apscheduler.schedulers.background import BackgroundScheduler


class SchedulerService:
    def __init__(self, scheduler: BackgroundScheduler) -> None:
        self._scheduler = scheduler

    def add_interval_job(
        self,
        job_id: str,
        func: Callable[..., object],
        seconds: int,
        *args: object,
    ) -> None:
        self._scheduler.add_job(
            func,
            trigger="interval",
            seconds=seconds,
            args=args,
            id=job_id,
            replace_existing=True,
        )

    def start(self) -> None:
        if not self._scheduler.running:
            self._scheduler.start()

    def stop(self) -> None:
        if self._scheduler.running:
            self._scheduler.shutdown(wait=False)

## Pydantic 数据校验工具

这个 notebook 演示如何用 Pydantic v2 编写可复用的数据校验工具，覆盖模型定义、字段约束、计算字段、批量校验、JSON 序列化、JSON Schema 和错误整理。代码中的类与函数保留 Google 风格中文 docstring，便于直接复制到项目模块中使用。

In [ ]:
"""Pydantic 数据校验工具示例。"""

from collections.abc import Mapping, Sequence
from decimal import Decimal
from typing import Any, Literal

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    TypeAdapter,
    ValidationError,
    computed_field,
    field_validator,
)

OrderStatus = Literal["pending", "paid", "cancelled"]


class OrderItem(BaseModel):
    """订单明细模型。

    Attributes:
        sku_id: 商品 SKU 编号。
        quantity: 购买数量，必须大于 0。
        unit_price: 商品单价，最多保留两位小数。
        line_total: 当前明细的金额小计。
    """

    model_config = ConfigDict(frozen=True, str_strip_whitespace=True)

    sku_id: str = Field(min_length=1, description="商品 SKU 编号")
    quantity: int = Field(gt=0, description="购买数量")
    unit_price: Decimal = Field(
        gt=Decimal("0"),
        max_digits=12,
        decimal_places=2,
        description="商品单价",
    )

    @field_validator("unit_price", mode="before")
    @classmethod
    def normalize_unit_price(cls, value: Any) -> Any:
        """规范化商品单价。

        Args:
            value: 用户输入的原始单价值。

        Returns:
            适合 Pydantic 继续解析的单价值。浮点数会先转成字符串，
            以减少二进制浮点表示带来的精度噪音。
        """
        if isinstance(value, float):
            return str(value)
        return value

    @computed_field
    @property
    def line_total(self) -> Decimal:
        """计算当前明细的小计金额。

        Returns:
            购买数量乘以商品单价后的金额。
        """
        return self.unit_price * self.quantity


class Order(BaseModel):
    """订单模型。

    Attributes:
        order_id: 订单编号。
        user_id: 用户编号。
        status: 订单状态，支持 pending、paid 和 cancelled。
        items: 订单明细列表，至少包含一条明细。
        total_amount: 订单总金额。
    """

    model_config = ConfigDict(extra="forbid", frozen=True, str_strip_whitespace=True)

    order_id: str = Field(min_length=1, description="订单编号")
    user_id: str = Field(min_length=1, description="用户编号")
    status: OrderStatus = Field(default="pending", description="订单状态")
    items: list[OrderItem] = Field(min_length=1, description="订单明细")

    @computed_field
    @property
    def total_amount(self) -> Decimal:
        """计算订单总金额。

        Returns:
            所有订单明细小计金额的总和。
        """
        return sum(item.line_total for item in self.items)


_ORDER_LIST_ADAPTER = TypeAdapter(list[Order])


def parse_order(payload: Mapping[str, Any] | str | bytes | bytearray) -> Order:
    """解析并校验单个订单。

    Args:
        payload: 订单字典、JSON 字符串或 JSON 字节串。

    Returns:
        校验通过后的订单模型。

    Raises:
        ValidationError: 当字段缺失、类型错误或业务约束不满足时抛出。
    """
    if isinstance(payload, str | bytes | bytearray):
        return Order.model_validate_json(payload)
    return Order.model_validate(payload)


def parse_orders(
    payloads: Sequence[Mapping[str, Any]] | str | bytes | bytearray,
) -> list[Order]:
    """批量解析并校验订单。

    Args:
        payloads: 订单字典序列、JSON 字符串或 JSON 字节串。

    Returns:
        校验通过后的订单模型列表。

    Raises:
        ValidationError: 当任意订单字段缺失、类型错误或业务约束不满足时抛出。
    """
    if isinstance(payloads, str | bytes | bytearray):
        return _ORDER_LIST_ADAPTER.validate_json(payloads)
    return _ORDER_LIST_ADAPTER.validate_python(payloads)


def dump_order(order: Order) -> dict[str, Any]:
    """把订单模型转换为适合 JSON 输出的字典。

    Args:
        order: 已校验的订单模型。

    Returns:
        JSON 友好的订单字典。Decimal 字段会转换成字符串，计算字段也会包含在结果中。
    """
    return order.model_dump(mode="json")


def order_schema() -> dict[str, Any]:
    """生成订单模型的 JSON Schema。

    Returns:
        可用于接口文档或前端表单生成的 JSON Schema 字典。
    """
    return Order.model_json_schema()


def summarize_validation_error(error: ValidationError) -> list[str]:
    """整理 Pydantic 校验错误。

    Args:
        error: Pydantic 抛出的校验异常。

    Returns:
        更适合日志或接口响应展示的错误消息列表。
    """
    messages: list[str] = []
    for item in error.errors():
        location = ".".join(str(part) for part in item["loc"])
        message = item["msg"]
        messages.append(f"{location}: {message}" if location else message)
    return messages

### 解析单个订单

Pydantic 会自动清理字符串首尾空白、校验字段约束，并生成 `line_total` 与 `total_amount` 这类计算字段。

In [ ]:
sample_order_payload = {
    "order_id": " ORD-001 ",
    "user_id": " USER-001 ",
    "status": "paid",
    "items": [
        {"sku_id": " SKU-1 ", "quantity": 2, "unit_price": "12.50"},
        {"sku_id": "SKU-2", "quantity": 1, "unit_price": 3.5},
    ],
}

order = parse_order(sample_order_payload)
dumped_order = dump_order(order)

assert order.order_id == "ORD-001"
assert order.items[0].sku_id == "SKU-1"
assert order.total_amount == Decimal("28.50")
assert dumped_order["total_amount"] == "28.50"

dumped_order

### 批量校验和生成 Schema

`TypeAdapter` 适合处理列表、联合类型等非 `BaseModel` 根类型；`model_json_schema()` 可以把模型约束转换成接口文档或表单系统可消费的结构。

In [ ]:
import json

raw_orders = json.dumps(
    [
        {
            "order_id": "ORD-002",
            "user_id": "USER-002",
            "items": [{"sku_id": "SKU-8", "quantity": 2, "unit_price": "19.90"}],
        }
    ],
    ensure_ascii=False,
)

orders = parse_orders(raw_orders)
schema = order_schema()

assert len(orders) == 1
assert orders[0].status == "pending"
assert orders[0].total_amount == Decimal("39.80")
assert schema["title"] == "Order"

schema["properties"]["items"]

### 整理校验错误

实际项目通常不会直接把 `ValidationError` 原样返回给调用方，可以先整理出稳定的字段路径和错误消息。

In [ ]:
try:
    parse_order(
        {
            "order_id": "ORD-003",
            "user_id": "USER-003",
            "items": [{"sku_id": "", "quantity": 0, "unit_price": "0"}],
        }
    )
except ValidationError as error:
    validation_messages = summarize_validation_error(error)

assert any(message.startswith("items.0.sku_id:") for message in validation_messages)
assert any(message.startswith("items.0.quantity:") for message in validation_messages)
assert any(message.startswith("items.0.unit_price:") for message in validation_messages)

validation_messages

## Gradio 和 Streamlit 选择

Gradio 和 Streamlit 都能把 Python 函数、数据分析流程和模型推理包装成浏览器里的交互应用。Gradio 更适合快速生成模型 Demo、输入输出组件和分享链接；Streamlit 更适合数据看板、内部工具和脚本式应用。

### 安装和启动方式

```bash
uv add gradio streamlit
```

Gradio 通常可以直接运行 Python 脚本或 notebook 代码单元。Streamlit 需要把代码保存为 `.py` 文件，然后用命令启动：

```bash
streamlit run app.py
```

### 怎么选

| 场景 | 优先选择 | 原因 |
| --- | --- | --- |
| 给模型函数做一个可分享 Demo | Gradio | `Interface` 能直接绑定函数、输入组件和输出组件 |
| 需要多个输入区域、事件流和布局 | Gradio | `Blocks` 更适合组合复杂交互 |
| 数据分析报告、指标看板、内部运营工具 | Streamlit | 脚本从上到下渲染，写法接近普通数据分析脚本 |
| 需要缓存数据读取或模型加载 | Streamlit | `st.cache_data` 和 `st.cache_resource` 适合分层缓存 |
| 需要多人协作迭代产品级前端 | 传统前端框架 | 这两个工具更偏快速原型和中后台工具 |

## 可测试的业务函数

无论用哪个框架，都应该先把业务逻辑写成普通 Python 函数。UI 只负责收集输入、调用函数、展示结果。

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class ReviewResult:
    label: str
    score: float
    reason: str


POSITIVE_WORDS = {"good", "great", "excellent", "amazing", "喜欢", "推荐", "满意"}
NEGATIVE_WORDS = {"bad", "poor", "terrible", "awful", "失望", "糟糕", "差"}


def analyze_review(text: str) -> ReviewResult:
    normalized = text.strip().lower()
    if not normalized:
        return ReviewResult(label="empty", score=0.0, reason="请输入一段评价文本。")

    positive_hits = sum(word in normalized for word in POSITIVE_WORDS)
    negative_hits = sum(word in normalized for word in NEGATIVE_WORDS)
    raw_score = positive_hits - negative_hits

    if raw_score > 0:
        label = "positive"
    elif raw_score < 0:
        label = "negative"
    else:
        label = "neutral"

    score = min(1.0, max(0.0, 0.5 + raw_score * 0.25))
    reason = f"positive={positive_hits}, negative={negative_hits}"
    return ReviewResult(label=label, score=score, reason=reason)


analyze_review("这个工具很好用，我很满意")

## Gradio

Gradio 适合把一个 Python 函数或模型推理流程快速包装成可交互 Demo。`Interface` 适合单函数输入输出，`Blocks` 适合多组件布局和事件绑定。

### Interface

`gr.Interface` 适合一个函数对应一组输入输出的场景。把函数、输入组件、输出组件和示例传进去，就能得到一个浏览器应用。

In [ ]:
def build_gradio_interface_code() -> str:
    return """
import gradio as gr


def analyze_review(text: str) -> dict[str, str | float]:
    normalized = text.strip().lower()
    positive_hits = sum(word in normalized for word in {"good", "great", "喜欢", "推荐", "满意"})
    negative_hits = sum(word in normalized for word in {"bad", "poor", "失望", "糟糕", "差"})
    raw_score = positive_hits - negative_hits
    label = "positive" if raw_score > 0 else "negative" if raw_score < 0 else "neutral"
    score = min(1.0, max(0.0, 0.5 + raw_score * 0.25))
    return {"label": label, "score": score, "reason": f"positive={positive_hits}, negative={negative_hits}"}


demo = gr.Interface(
    fn=analyze_review,
    inputs=gr.Textbox(lines=4, label="Review"),
    outputs=gr.JSON(label="Result"),
    examples=["这个工具很好用，我很满意", "The experience was poor and bad"],
    title="Review Analyzer",
)


if __name__ == "__main__":
    demo.launch()
""".strip()


print(build_gradio_interface_code())

### Blocks

`Blocks` 适合多个组件、多个按钮、分栏布局和事件回调。常见结构是：创建组件，定义事件，把按钮点击或输入变化绑定到函数。

In [ ]:
def build_gradio_blocks_code() -> str:
    return """
import gradio as gr


def summarize(text: str, max_words: int) -> str:
    words = text.split()
    if not words:
        return "请输入需要摘要的文本。"
    summary = " ".join(words[:max_words])
    suffix = "..." if len(words) > max_words else ""
    return f"{summary}{suffix}"


with gr.Blocks(title="Text Helper") as demo:
    gr.Markdown("# Text Helper")
    with gr.Row():
        text = gr.Textbox(lines=8, label="Input")
        output = gr.Textbox(lines=8, label="Summary")
    max_words = gr.Slider(5, 50, value=20, step=1, label="Max words")
    submit = gr.Button("Summarize", variant="primary")

    submit.click(fn=summarize, inputs=[text, max_words], outputs=output)


if __name__ == "__main__":
    demo.launch()
""".strip()


print(build_gradio_blocks_code())

## Streamlit

Streamlit 应用就是一个从上到下执行的 Python 脚本。用户改动控件时，脚本会重新运行，因此耗时的数据加载和模型初始化要做缓存。

In [ ]:
def build_streamlit_app_code() -> str:
    return """
import pandas as pd
import streamlit as st


st.set_page_config(page_title="Sales Dashboard", layout="wide")
st.title("Sales Dashboard")


@st.cache_data
def load_sales() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"region": "East", "product": "A", "revenue": 1200},
            {"region": "East", "product": "B", "revenue": 900},
            {"region": "West", "product": "A", "revenue": 1600},
            {"region": "West", "product": "B", "revenue": 700},
        ]
    )


sales = load_sales()
regions = st.sidebar.multiselect("Region", sorted(sales["region"].unique()), default=sorted(sales["region"].unique()))
filtered = sales[sales["region"].isin(regions)]

total_revenue = int(filtered["revenue"].sum())
st.metric("Total revenue", f"${total_revenue:,}")

left, right = st.columns(2)
with left:
    st.subheader("Raw data")
    st.dataframe(filtered, use_container_width=True)

with right:
    st.subheader("Revenue by product")
    chart_data = filtered.groupby("product", as_index=False)["revenue"].sum()
    st.bar_chart(chart_data, x="product", y="revenue")
""".strip()


print(build_streamlit_app_code())

### 状态和缓存

`st.session_state` 保存当前浏览器会话里的状态，适合计数器、聊天历史、筛选条件等用户级状态。`st.cache_data` 适合缓存可序列化数据结果，`st.cache_resource` 适合缓存数据库连接、模型对象这类资源。

In [ ]:
def build_streamlit_state_code() -> str:
    return """
from dataclasses import dataclass

import streamlit as st


@dataclass
class TinyModel:
    name: str

    def predict(self, text: str) -> str:
        return f"{self.name}: {len(text)} chars"


@st.cache_resource
def load_model() -> TinyModel:
    return TinyModel(name="demo-model")


model = load_model()
st.title("Session State Demo")

if "history" not in st.session_state:
    st.session_state.history = []

text = st.text_input("Text")
if st.button("Predict") and text:
    result = model.predict(text)
    st.session_state.history.append({"text": text, "result": result})

st.write(st.session_state.history)
""".strip()


print(build_streamlit_state_code())

## Python Web 应用项目结构

Demo 可以只有一个文件，但真实项目建议把业务逻辑和 UI 分开：

```text
my_app/
├── app.py              # Streamlit 或 Gradio 入口
├── services.py         # 数据读取、模型推理、外部 API
├── schemas.py          # dataclass / pydantic 模型
├── tests/
│   └── test_services.py
└── README.md
```

这样可以单独测试 `services.py`，UI 层只保留组件、布局和事件绑定。

### 常见注意点

- 不要在 UI 回调里硬编码密钥，使用环境变量或托管平台的 secrets 配置。
- 不要把耗时初始化放在 Streamlit 顶层反复执行，优先使用缓存装饰器。
- Gradio 回调函数应该返回和输出组件匹配的数据类型，例如字符串、字典、图片路径或数组。
- 文件上传、图片处理和模型推理要限制大小，必要时做超时和异常处理。
- 原型可直接用内置服务启动；需要团队长期使用时，再考虑认证、日志、监控和反向代理。

## Playwright 浏览器自动化

Playwright 是现代端到端测试和浏览器自动化工具，支持 Chromium、Firefox 和 WebKit。这里优先讲定位器、自动等待、断言、网络拦截和截图这些更贴近现代 Web 测试的能力。

### 安装和浏览器

安装 Python 包后，还需要下载浏览器二进制。命令行示例：

```zsh
uv add playwright
uv run playwright install
```

如果写端到端测试，官方推荐配合 `pytest-playwright`；如果只是脚本化浏览器自动化，直接使用 `playwright.sync_api` 或 `playwright.async_api` 即可。

### 同步 API：打开页面并断言

Playwright 的定位器会自动等待元素变为可操作状态，通常不需要手写 `sleep()`。

In [ ]:
import re

from playwright.sync_api import expect, sync_playwright


with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://playwright.dev/python/")

    expect(page).to_have_title(re.compile("Playwright"))
    expect(page.get_by_role("link", name="Docs")).to_be_visible()

    browser.close()

### 定位器优先级

现代测试优先使用用户可感知的定位方式，例如 role、label、text 和 test id。只有在页面没有语义信息时，才退回 CSS 或 XPath。

In [ ]:
from playwright.sync_api import expect, sync_playwright


with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://demo.playwright.dev/todomvc")

    page.get_by_placeholder("What needs to be done?").fill("Learn Playwright")
    page.keyboard.press("Enter")

    expect(page.get_by_text("Learn Playwright")).to_be_visible()
    browser.close()

### 异步 API

如果项目本身使用 `asyncio`，优先使用异步 API，避免在线程池或事件循环之间来回切换。

In [ ]:
import asyncio

from playwright.async_api import async_playwright, expect


async def main() -> None:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto("https://playwright.dev/python/")
        await expect(
            page.get_by_role("heading", name="Playwright for Python")
        ).to_be_visible()
        await browser.close()


asyncio.run(main())

### 网络拦截

Playwright 可以拦截请求，常用于替换第三方接口、屏蔽慢资源或让测试数据稳定。

In [ ]:
from playwright.sync_api import Route, sync_playwright


def block_images(route: Route) -> None:
    if route.request.resource_type == "image":
        route.abort()
    else:
        route.continue_()


with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.route("**/*", block_images)
    page.goto("https://playwright.dev/python/")
    print(page.title())
    browser.close()

### 截图和调试

截图、trace viewer 和 codegen 能显著减少调试成本。脚本中可以用 `page.screenshot()` 保存状态；测试中可以开启 trace，用 `playwright show-trace` 回放。

In [ ]:
from pathlib import Path

from playwright.sync_api import sync_playwright


output = Path("playwright-home.png")
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(viewport={"width": 1280, "height": 720})
    page.goto("https://playwright.dev/python/")
    page.screenshot(path=output, full_page=True)
    browser.close()

print(output)